# 🔬 ECE-226: Roofline Model & LLM Kernel Benchmarking
## Benchmarking Methodologies & Profiling Ecosystems

---

### 📌 What this notebook does (read this first!)

This notebook measures three fundamental AI operations (kernels) used inside Transformer/LLM models,
and plots them on a **Roofline Model** — a graph that shows where each operation sits relative to the
hardware's physical limits.

**The three kernels we study:**
| Kernel | What it represents in an LLM | Expected behavior |
|--------|------------------------------|-------------------|
| **GEMM** (Matrix Multiply) | Q/K/V projections, MLP layers | Compute-bound at large sizes |
| **Element-wise Add** | Residual connections | Always memory-bound |
| **Softmax** | Attention score normalization | Memory-bound |

**What we produce:**
1. 📋 GPU Hardware Specification Table
2. 📈 Theoretical Roofline Plot (FP32 + FP16)
3. ⏱️ Latency vs Size plots for all kernels
4. 🔢 Operational Intensity (OI) vs Size plots
5. 📊 Achieved GFLOPs vs Size plots
6. 🗺️ Final Roofline Map with all kernels plotted
7. 📉 FP32 vs FP16 comparative analysis
8. 🏷️ Bound classification table
9. 🔍 PyTorch Profiler trace analysis

---
**➡️ Run cells top to bottom. Each section is explained before the code.**

## Section 0 — Install & Import Everything

We import PyTorch (for running GPU operations), matplotlib (for graphs), pandas (for tables), and numpy (for math).

In [ ]:
# ── Install any missing packages (Colab usually has all of these) ──
!pip install tabulate --quiet

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
import warnings
import time
import os
from tabulate import tabulate
from torch.profiler import profile, record_function, ProfilerActivity

warnings.filterwarnings('ignore')

# ── Reproducibility settings (as specified in team's shared specs) ──
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# ── Verify GPU is available ──
assert torch.cuda.is_available(), "❌ No GPU found! Go to Runtime > Change runtime type > GPU"
print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA version: {torch.version.cuda}")

---
## Section 1 — Hardware Characterization

### Why this matters
Before measuring anything, we need to know the **theoretical limits** of our hardware.
Every GPU has two fundamental limits:
- **Peak Compute** — the maximum number of math operations it can do per second (measured in TFLOP/s)
- **Peak Memory Bandwidth** — the maximum speed at which it can read/write data from GPU memory (measured in GB/s)

These two limits define the "roof" in the Roofline Model.

We use a lookup table of known GPU specs. If your GPU isn't in the table, we fall back to a safe estimate.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# GPU Specifications Database
# Source: NVIDIA official spec sheets
# FP32 and FP16 are in TFLOP/s, Bandwidth in GB/s
# ═══════════════════════════════════════════════════════════════════
GPU_SPECS = {
    # Colab GPUs
    "NVIDIA L4":              {"fp32": 30.3,  "fp16": 242.0, "bw": 300.0,  "vram_gb": 24},
    "Tesla T4":               {"fp32": 8.1,   "fp16": 65.0,  "bw": 300.0,  "vram_gb": 16},
    "Tesla V100-SXM2-16GB":   {"fp32": 15.7,  "fp16": 125.0, "bw": 900.0,  "vram_gb": 16},
    "Tesla V100-PCIE-16GB":   {"fp32": 14.1,  "fp16": 112.0, "bw": 897.0,  "vram_gb": 16},
    "NVIDIA A100-SXM4-40GB":  {"fp32": 19.5,  "fp16": 312.0, "bw": 1555.0, "vram_gb": 40},
    "NVIDIA A100-SXM4-80GB":  {"fp32": 19.5,  "fp16": 312.0, "bw": 2000.0, "vram_gb": 80},
    "NVIDIA A100-PCIE":       {"fp32": 19.5,  "fp16": 312.0, "bw": 1935.0, "vram_gb": 40},
    # Consumer GPUs
    "NVIDIA GeForce RTX 3080": {"fp32": 29.8,  "fp16": 29.8,  "bw": 760.0,  "vram_gb": 10},
    "NVIDIA GeForce RTX 3090": {"fp32": 35.6,  "fp16": 35.6,  "bw": 936.0,  "vram_gb": 24},
    "NVIDIA GeForce RTX 4090": {"fp32": 82.6,  "fp16": 165.2, "bw": 1008.0, "vram_gb": 24},
    "NVIDIA GeForce RTX 4080": {"fp32": 48.7,  "fp16": 97.5,  "bw": 736.0,  "vram_gb": 16},
    "NVIDIA GeForce RTX 5070 Ti": {"fp32": 43.0, "fp16": 86.0, "bw": 896.0, "vram_gb": 16},
    # Datahub GPU
    "NVIDIA GeForce GTX 1080 Ti": {"fp32": 11.3, "fp16": 0.18, "bw": 484.0, "vram_gb": 11},
}

# ── Auto-detect GPU and look up specs ──
gpu_name = torch.cuda.get_device_name(0)
print(f"Detected GPU: '{gpu_name}'")

# Try exact match, then partial match
specs = GPU_SPECS.get(gpu_name)
if specs is None:
    for key in GPU_SPECS:
        if key.lower() in gpu_name.lower() or gpu_name.lower() in key.lower():
            specs = GPU_SPECS[key]
            print(f"  Matched to known GPU: '{key}'")
            break

if specs is None:
    print("  ⚠️  GPU not in database. Using safe fallback estimates.")
    print("  (For accuracy, add your GPU specs to GPU_SPECS dict above)")
    # Fallback: measure VRAM and use conservative estimates
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    specs = {"fp32": 10.0, "fp16": 20.0, "bw": 300.0, "vram_gb": round(vram, 1)}

PEAK_FP32_TFLOPS = specs["fp32"]   # Theoretical peak FP32 compute
PEAK_FP16_TFLOPS = specs["fp16"]   # Theoretical peak FP16 compute (Tensor Cores)
PEAK_BW_GBs      = specs["bw"]     # Peak memory bandwidth
VRAM_GB          = specs["vram_gb"]

# Also grab runtime properties
props = torch.cuda.get_device_properties(0)
NUM_SMS       = props.multi_processor_count
SHARED_MEM_KB = props.shared_memory_per_block // 1024
TOTAL_MEM_GB  = props.total_memory / 1e9

# ── Print a clean hardware specification table ──
print("\n" + "═"*55)
print("         🖥️  GPU HARDWARE SPECIFICATION TABLE")
print("═"*55)
hw_data = [
    ["GPU Model",                    gpu_name],
    ["VRAM",                         f"{TOTAL_MEM_GB:.1f} GB"],
    ["Number of SMs (Streaming Multiprocessors)", str(NUM_SMS)],
    ["Shared Memory per Block",      f"{SHARED_MEM_KB} KB"],
    ["Peak FP32 Compute",            f"{PEAK_FP32_TFLOPS:.1f} TFLOP/s"],
    ["Peak FP16 Compute (Tensor Core)", f"{PEAK_FP16_TFLOPS:.1f} TFLOP/s"],
    ["FP16 Speedup over FP32",       f"{PEAK_FP16_TFLOPS/PEAK_FP32_TFLOPS:.1f}x"],
    ["Peak Memory Bandwidth",        f"{PEAK_BW_GBs:.1f} GB/s"],
    ["PyTorch Version",              torch.__version__],
    ["CUDA Version",                 torch.version.cuda],
]
print(tabulate(hw_data, headers=["Parameter", "Value"], tablefmt="grid"))
print()

---
## Section 2 — Construct the Theoretical Roofline Model

### What is the Roofline Model?

The Roofline Model is a graph with two axes:
- **X-axis:** Operational Intensity (OI) = FLOPs ÷ Bytes — "how much math does this operation do per byte of data it reads?"
- **Y-axis:** Performance = FLOPs ÷ Time — "how fast does it run?"

The "roofline" is defined by two limits:
1. **Memory bandwidth roof:** If OI is low, performance = OI × Bandwidth (you're memory-limited)
2. **Compute roof:** If OI is high, performance = Peak Compute (you're compute-limited)

Combined: `Performance ≤ min(Peak_Compute, OI × Peak_Bandwidth)`

We plot both FP32 and FP16 rooflines since FP16 has a much higher compute ceiling.

In [ ]:
def plot_roofline(ax, peak_compute_tflops, peak_bw_gbs, label, color, linestyle='-'):
    """
    Draw a single roofline on an axes object.
    
    The roofline has two segments:
    1. Memory-bound region: performance = OI * bandwidth  (slanted line)
    2. Compute-bound region: performance = peak_compute   (flat line)
    The intersection point ("ridge point") tells you where the transition happens.
    """
    peak_compute_gflops = peak_compute_tflops * 1000  # Convert TFLOP/s → GFLOP/s
    
    # Ridge point: OI where memory line meets compute ceiling
    # At ridge: OI * BW = Peak_Compute  →  OI_ridge = Peak_Compute / BW
    ridge_oi = peak_compute_gflops / peak_bw_gbs
    
    oi_range = np.logspace(-2, 4, 500)  # OI from 0.01 to 10000
    
    # Roofline formula: min(compute_ceiling, memory_line)
    performance = np.minimum(peak_compute_gflops, oi_range * peak_bw_gbs)
    
    ax.plot(oi_range, performance, color=color, linestyle=linestyle,
            linewidth=2.5, label=f"{label} (Peak: {peak_compute_tflops:.0f} TFLOP/s)")
    
    # Mark the ridge point
    ax.axvline(x=ridge_oi, color=color, linestyle=':', alpha=0.4, linewidth=1)
    ax.annotate(f'Ridge\n{ridge_oi:.1f}', xy=(ridge_oi, peak_compute_gflops),
                xytext=(ridge_oi*1.3, peak_compute_gflops*0.7),
                fontsize=7, color=color, alpha=0.8,
                arrowprops=dict(arrowstyle='->', color=color, lw=1))
    
    return ridge_oi


fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Theoretical Roofline Model\n{gpu_name}', fontsize=14, fontweight='bold')

for ax, title, compute, color_mem, color_comp in [
    (axes[0], "FP32 Roofline", PEAK_FP32_TFLOPS, '#2196F3', '#F44336'),
    (axes[1], "FP16 Roofline", PEAK_FP16_TFLOPS, '#4CAF50', '#FF9800'),
]:
    ax.set_xscale('log')
    ax.set_yscale('log')
    
    # Plot memory bandwidth line separately for annotation
    oi_range = np.logspace(-2, 4, 500)
    ridge_oi = (compute * 1000) / PEAK_BW_GBs
    
    # Memory-bound segment
    oi_mem = oi_range[oi_range <= ridge_oi]
    ax.plot(oi_mem, oi_mem * PEAK_BW_GBs, color=color_mem, linewidth=3,
            label=f'Memory Bandwidth Roof\n({PEAK_BW_GBs:.0f} GB/s)')
    
    # Compute-bound segment
    oi_comp = oi_range[oi_range >= ridge_oi]
    ax.plot(oi_comp, np.full_like(oi_comp, compute * 1000), color=color_comp, linewidth=3,
            label=f'Compute Roof ({compute:.0f} TFLOP/s)')
    
    # Shade regions
    ax.fill_between(oi_range, np.minimum(compute*1000, oi_range*PEAK_BW_GBs),
                    alpha=0.05, color='gray', label='Attainable Region')
    
    # Ridge point
    ax.axvline(x=ridge_oi, color='black', linestyle='--', alpha=0.5, linewidth=1.5)
    ax.annotate(f'Ridge Point\nOI = {ridge_oi:.1f}\nFLOP/Byte',
                xy=(ridge_oi, compute * 500),
                xytext=(ridge_oi * 3, compute * 200),
                fontsize=8, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5),
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))
    
    # Labels and formatting
    ax.set_xlabel('Operational Intensity (FLOP/Byte)', fontsize=11)
    ax.set_ylabel('Achieved Performance (GFLOP/s)', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(True, which='both', alpha=0.3)
    ax.set_xlim([1e-2, 1e4])
    ax.set_ylim([1, compute * 3000])
    
    # Annotate regions
    ax.text(0.03, 0.55, 'MEMORY\nBOUND', transform=ax.transAxes,
            fontsize=10, color=color_mem, fontweight='bold', alpha=0.7)
    ax.text(0.72, 0.55, 'COMPUTE\nBOUND', transform=ax.transAxes,
            fontsize=10, color=color_comp, fontweight='bold', alpha=0.7)

plt.tight_layout()
plt.savefig('roofline_theoretical.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Roofline plots saved as 'roofline_theoretical.png'")
print(f"\n📌 Key insight: FP16 compute ceiling is {PEAK_FP16_TFLOPS/PEAK_FP32_TFLOPS:.0f}x higher than FP32,")
print(f"   but the memory bandwidth roof ({PEAK_BW_GBs} GB/s) stays the SAME.")
print(f"   This means FP16 only helps compute-bound kernels, NOT memory-bound ones.")

---
## Section 3 — Define Kernels & FLOPs/Memory Formulas

### Why do we calculate FLOPs and memory bytes manually?

To place a kernel on the roofline, we need:
- **FLOPs** — how many math operations does this kernel do?
- **Memory bytes** — how many bytes does it read/write from GPU memory?
- **Operational Intensity (OI)** = FLOPs ÷ Bytes
- **Achieved Performance** = FLOPs ÷ Time

These formulas are standardized by the team specs document:

| Kernel | FLOPs formula | Memory formula |
|--------|--------------|----------------|
| GEMM | `2 * B * N * D²` | `dtype_bytes * (B*N*D + D*D + B*N*D)` |
| Element-wise Add | `B * N * D` | `dtype_bytes * (A + B + C) = 3 * B*N*D` |
| Softmax | `B * N * 5N` | `dtype_bytes * (2 * B*N*N)` |

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# KERNEL DEFINITIONS — as specified in team's shared specs
# B = batch size, N = sequence length, D = hidden dimension
# ═══════════════════════════════════════════════════════════════════

def run_gemm(B, N, D, dtype):
    """GEMM: represents Q/K/V projections and MLP layers in Transformers.
    A shape: (B, N, D)  →  reshaped to (B*N, D)
    B shape: (D, D)
    """
    A = torch.randn(B * N, D, dtype=dtype, device='cuda')
    W = torch.randn(D, D, dtype=dtype, device='cuda')
    with torch.no_grad():
        out = torch.matmul(A, W)
    return out

def run_elementwise_add(B, N, D, dtype):
    """Element-wise Add: represents residual connections / skip connections.
    Shape: (B, N, D) + (B, N, D)
    """
    A = torch.randn(B, N, D, dtype=dtype, device='cuda')
    Bm = torch.randn(B, N, D, dtype=dtype, device='cuda')
    with torch.no_grad():
        out = torch.add(A, Bm)
    return out

def run_softmax(B, N, dtype):
    """Softmax: represents attention score normalization.
    Shape: (B, N, N)  — attention matrix
    """
    A = torch.randn(B, N, N, dtype=dtype, device='cuda')
    with torch.no_grad():
        out = torch.softmax(A, dim=-1)
    return out


# ── FLOPs & Memory Traffic Formulas ──
def gemm_flops(B, N, D):
    """FLOPs = 2 * B * N * D^2  (each output element = D multiplications + D additions)"""
    return 2 * B * N * D * D

def gemm_bytes(B, N, D, dtype_bytes):
    """Bytes = dtype_bytes * (input A + weight W + output C)"""
    return dtype_bytes * (B * N * D  +  D * D  +  B * N * D)

def add_flops(B, N, D):
    """FLOPs = B * N * D  (one addition per element)"""
    return B * N * D

def add_bytes(B, N, D, dtype_bytes):
    """Bytes = dtype_bytes * 3 * B * N * D  (read A, read B, write C)"""
    return dtype_bytes * 3 * B * N * D

def softmax_flops(B, N):
    """FLOPs ≈ 5 * N per row (max, subtract, exp, sum, divide) * B*N rows"""
    return B * N * 5 * N

def softmax_bytes(B, N, dtype_bytes):
    """Bytes = dtype_bytes * 2 * B * N * N  (read input, write output)"""
    return dtype_bytes * 2 * B * N * N


print("✅ Kernel functions and FLOPs/Memory formulas defined.")
print("\n📐 Formula summary:")
formulas = [
    ["GEMM",          "2 × B × N × D²",    "dtype_bytes × (B×N×D + D² + B×N×D)"],
    ["Element-wise Add", "B × N × D",       "dtype_bytes × 3 × B × N × D"],
    ["Softmax",        "B × N × 5N",        "dtype_bytes × 2 × B × N × N"],
]
print(tabulate(formulas, headers=["Kernel", "FLOPs", "Memory Bytes"], tablefmt="grid"))

---
## Section 4 — Benchmarking Harness

### Why can't we just use Python's `time.time()`?

The GPU runs **asynchronously** from the CPU. When you call `torch.matmul(A, B)`, Python returns immediately — but the GPU hasn't finished yet! If you measure time without waiting for the GPU to finish, you'll record near-zero latency which is completely wrong.

**The fix:** Use `torch.cuda.synchronize()` which forces Python to wait until the GPU is done before recording the stop time.

We also do:
- **20 warmup iterations** — first runs are slower (GPU caches cold, CUDA kernels compiling)
- **100 timed iterations** — average over many runs for stability
- **3 experiment repeats** — to check consistency and measure std deviation

In [ ]:
def benchmark_kernel(kernel_fn, warmup=20, timed=100, repeats=3):
    """
    Properly measure GPU kernel latency.
    
    Args:
        kernel_fn: a callable (no args) that runs the GPU operation
        warmup:    how many warmup iterations (excluded from timing)
        timed:     how many iterations to time
        repeats:   how many full experiments to run (for mean/std)
    
    Returns:
        mean_ms, std_ms: mean and std of latency in milliseconds
    """
    # === WARMUP PHASE ===
    # GPU starts cold: first iterations include kernel compilation,
    # cache warming, and CUDA auto-tuning. We throw these away.
    with torch.no_grad():
        for _ in range(warmup):
            kernel_fn()
    torch.cuda.synchronize()  # Wait for all warmup to finish
    
    # === TIMED PHASE ===
    experiment_times = []
    for rep in range(repeats):
        torch.cuda.synchronize()         # Ensure GPU is idle before we start timing
        t_start = time.perf_counter()    # High-precision CPU timer
        
        with torch.no_grad():
            for _ in range(timed):
                kernel_fn()
        
        torch.cuda.synchronize()         # ← CRITICAL: wait for GPU to actually finish
        t_end = time.perf_counter()
        
        total_ms = (t_end - t_start) * 1000  # Convert seconds → milliseconds
        avg_ms   = total_ms / timed           # Per-iteration latency
        experiment_times.append(avg_ms)
    
    return np.mean(experiment_times), np.std(experiment_times)


print("✅ Benchmarking harness defined.")
print("\n📋 Experimental Control Checklist:")
checklist = [
    ["✅", "Warmup iterations",           "20 (throws away cold-start overhead)"],
    ["✅", "Timed iterations",            "100 (averages out noise)"],
    ["✅", "GPU synchronization",         "torch.cuda.synchronize() before & after"],
    ["✅", "No-gradient context",         "torch.no_grad() (no backprop overhead)"],
    ["✅", "Experiment repeats",          "3 (for mean ± std)"],
    ["✅", "Determinism seeds",           "torch.manual_seed(42)"],
    ["✅", "cuDNN determinism",           "benchmark=False, deterministic=True"],
    ["✅", "Fixed batch size",            "B=1 per spec"],
]
print(tabulate(checklist, headers=["Done", "Control", "Description"], tablefmt="grid"))

---
## Section 5 — Run Size & Precision Sweep

We now run all three kernels across:
- **5 sizes:** D = N = 256, 512, 1024, 2048, 4096
- **2 precisions:** FP32 (standard) and FP16 (half-precision, faster on modern GPUs)

For each combination, we record: latency, FLOPs, bytes, OI, and achieved GFLOPs.

**⏳ This cell takes 3–8 minutes to run. That's normal.**

In [ ]:
# ── Sweep parameters (from team's shared specs) ──
SIZES      = [256, 512, 1024, 2048, 4096]
PRECISIONS = {'FP32': (torch.float32, 4),   # dtype, bytes_per_element
              'FP16': (torch.float16, 2)}
BATCH_SIZE = 1  # B=1 as per shared specs

results = []   # Will collect all rows here

KERNELS = [
    ('GEMM',             'compute-bound (expected)',  '#E74C3C'),
    ('Element-wise Add', 'memory-bound (expected)',   '#3498DB'),
    ('Softmax',          'memory-bound (expected)',   '#2ECC71'),
]

total_runs = len(SIZES) * len(PRECISIONS) * len(KERNELS)
run_idx = 0

print(f"Running {total_runs} experiments (3 kernels × 5 sizes × 2 precisions)...\n")
print(f"{'Kernel':<22} {'Prec':<5} {'D=N':<6} {'Latency (ms)':<14} {'OI (FLOP/B)':<14} {'GFLOPs':<10}")
print("-" * 75)

for prec_name, (dtype, dtype_bytes) in PRECISIONS.items():
    for D in SIZES:
        N = D   # sequence length = hidden dim (as per spec)
        B = BATCH_SIZE
        
        # ── GEMM ──
        flops = gemm_flops(B, N, D)
        mem   = gemm_bytes(B, N, D, dtype_bytes)
        oi    = flops / mem
        fn    = lambda: run_gemm(B, N, D, dtype)
        lat_mean, lat_std = benchmark_kernel(fn)
        gflops = (flops / 1e9) / (lat_mean / 1000)
        results.append({'kernel': 'GEMM', 'precision': prec_name, 'batch': B,
                        'hidden_dim': D, 'seq_len': N, 'latency_ms': lat_mean,
                        'latency_std': lat_std, 'flops': flops, 'memory_bytes': mem,
                        'operational_intensity': oi, 'achieved_gflops': gflops})
        run_idx += 1
        print(f"{'GEMM':<22} {prec_name:<5} {D:<6} {lat_mean:<14.3f} {oi:<14.2f} {gflops:<10.1f}")
        
        # ── Element-wise Add ──
        flops = add_flops(B, N, D)
        mem   = add_bytes(B, N, D, dtype_bytes)
        oi    = flops / mem
        fn    = lambda: run_elementwise_add(B, N, D, dtype)
        lat_mean, lat_std = benchmark_kernel(fn)
        gflops = (flops / 1e9) / (lat_mean / 1000)
        results.append({'kernel': 'Element-wise Add', 'precision': prec_name, 'batch': B,
                        'hidden_dim': D, 'seq_len': N, 'latency_ms': lat_mean,
                        'latency_std': lat_std, 'flops': flops, 'memory_bytes': mem,
                        'operational_intensity': oi, 'achieved_gflops': gflops})
        run_idx += 1
        print(f"{'Element-wise Add':<22} {prec_name:<5} {D:<6} {lat_mean:<14.3f} {oi:<14.4f} {gflops:<10.1f}")
        
        # ── Softmax ──
        flops = softmax_flops(B, N)
        mem   = softmax_bytes(B, N, dtype_bytes)
        oi    = flops / mem
        fn    = lambda: run_softmax(B, N, dtype)
        lat_mean, lat_std = benchmark_kernel(fn)
        gflops = (flops / 1e9) / (lat_mean / 1000)
        results.append({'kernel': 'Softmax', 'precision': prec_name, 'batch': B,
                        'hidden_dim': D, 'seq_len': N, 'latency_ms': lat_mean,
                        'latency_std': lat_std, 'flops': flops, 'memory_bytes': mem,
                        'operational_intensity': oi, 'achieved_gflops': gflops})
        run_idx += 1
        print(f"{'Softmax':<22} {prec_name:<5} {D:<6} {lat_mean:<14.3f} {oi:<14.4f} {gflops:<10.1f}")
    
    print()  # blank line between FP32 and FP16 blocks

# Convert to DataFrame for easy analysis
df = pd.DataFrame(results)
df.to_csv('benchmark_results.csv', index=False)
print("\n✅ All experiments complete! Results saved to 'benchmark_results.csv'")

---
## Section 6 — Full Results Table

Display the complete standardized results in the CSV format specified by the team.

In [ ]:
print("═" * 110)
print("                        📊 COMPLETE BENCHMARK RESULTS TABLE")
print("═" * 110)

display_df = df[['kernel', 'precision', 'hidden_dim', 'latency_ms', 
                 'flops', 'memory_bytes', 'operational_intensity', 'achieved_gflops']].copy()

display_df['flops']         = display_df['flops'].apply(lambda x: f"{x:.2e}")
display_df['memory_bytes']  = display_df['memory_bytes'].apply(lambda x: f"{x:.2e}")
display_df['latency_ms']    = display_df['latency_ms'].apply(lambda x: f"{x:.4f}")
display_df['operational_intensity'] = display_df['operational_intensity'].apply(lambda x: f"{x:.4f}")
display_df['achieved_gflops'] = display_df['achieved_gflops'].apply(lambda x: f"{x:.2f}")

display_df.columns = ['Kernel', 'Prec', 'D=N', 'Latency (ms)', 
                       'FLOPs', 'Mem Bytes', 'OI (FLOP/B)', 'GFLOPs']

print(tabulate(display_df, headers='keys', tablefmt='grid', showindex=False))

---
## Section 7 — Latency vs Size Plots

These plots show how each kernel's latency grows as we increase the problem size.
- Faster growth = the kernel scales more poorly with size
- Comparing FP32 vs FP16 shows how much speedup we get from half-precision

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('⏱️ Latency vs Problem Size — FP32 vs FP16', fontsize=14, fontweight='bold')

kernel_names  = ['GEMM', 'Element-wise Add', 'Softmax']
kernel_colors = {'FP32': '#E74C3C', 'FP16': '#3498DB'}
markers       = {'FP32': 'o', 'FP16': 's'}

for ax, kname in zip(axes, kernel_names):
    kdf = df[df['kernel'] == kname]
    
    for prec in ['FP32', 'FP16']:
        sub = kdf[kdf['precision'] == prec].sort_values('hidden_dim')
        ax.plot(sub['hidden_dim'], sub['latency_ms'],
                color=kernel_colors[prec], marker=markers[prec],
                linewidth=2, markersize=8, label=f'{prec}')
        # Error bars if std is available
        ax.fill_between(sub['hidden_dim'],
                        sub['latency_ms'] - sub['latency_std'],
                        sub['latency_ms'] + sub['latency_std'],
                        alpha=0.2, color=kernel_colors[prec])
    
    ax.set_xlabel('Hidden Dimension D (= Sequence Length N)', fontsize=10)
    ax.set_ylabel('Latency (ms)', fontsize=10)
    ax.set_title(f'{kname}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(SIZES)
    ax.set_xticklabels(SIZES, rotation=45)

plt.tight_layout()
plt.savefig('latency_vs_size.png', dpi=150, bbox_inches='tight')
plt.show()

# Print FP16 speedup ratios
print("\n📊 FP16 Speedup over FP32 (Latency Ratio):")
speedup_data = []
for kname in kernel_names:
    row = [kname]
    for D in SIZES:
        fp32_lat = df[(df['kernel']==kname)&(df['precision']=='FP32')&(df['hidden_dim']==D)]['latency_ms'].values[0]
        fp16_lat = df[(df['kernel']==kname)&(df['precision']=='FP16')&(df['hidden_dim']==D)]['latency_ms'].values[0]
        row.append(f"{fp32_lat/fp16_lat:.2f}x")
    speedup_data.append(row)
print(tabulate(speedup_data, headers=['Kernel'] + [f'D={d}' for d in SIZES], tablefmt='grid'))

---
## Section 8 — Operational Intensity (OI) vs Size Plots

OI = FLOPs ÷ Bytes. This is the most important metric for the roofline model.

- **High OI** → kernel does lots of math per byte loaded → can be compute-bound
- **Low OI** → kernel loads lots of data but does little math → memory-bound

We also draw the **ridge point line** — kernels with OI above this line are compute-bound, below are memory-bound.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('🔢 Operational Intensity vs Problem Size', fontsize=14, fontweight='bold')

ridge_fp32 = (PEAK_FP32_TFLOPS * 1000) / PEAK_BW_GBs
ridge_fp16 = (PEAK_FP16_TFLOPS * 1000) / PEAK_BW_GBs

for ax, kname in zip(axes, kernel_names):
    kdf = df[df['kernel'] == kname]
    
    for prec, color in kernel_colors.items():
        sub = kdf[kdf['precision'] == prec].sort_values('hidden_dim')
        ax.plot(sub['hidden_dim'], sub['operational_intensity'],
                color=color, marker=markers[prec], linewidth=2, markersize=8, label=prec)
    
    # Draw ridge point lines
    ax.axhline(y=ridge_fp32, color='#E74C3C', linestyle='--', alpha=0.6, linewidth=1.5,
               label=f'FP32 Ridge ({ridge_fp32:.1f})')
    ax.axhline(y=ridge_fp16, color='#3498DB', linestyle='--', alpha=0.6, linewidth=1.5,
               label=f'FP16 Ridge ({ridge_fp16:.1f})')
    
    ax.set_xlabel('Hidden Dimension D', fontsize=10)
    ax.set_ylabel('Operational Intensity (FLOP/Byte)', fontsize=10)
    ax.set_title(f'{kname}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(SIZES)
    ax.set_xticklabels(SIZES, rotation=45)
    ax.set_yscale('log')

plt.tight_layout()
plt.savefig('oi_vs_size.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📌 Ridge points (OI threshold between memory-bound and compute-bound):")
print(f"   FP32 ridge: {ridge_fp32:.2f} FLOP/Byte")
print(f"   FP16 ridge: {ridge_fp16:.2f} FLOP/Byte")
print(f"   Kernels with OI < ridge → MEMORY BOUND")
print(f"   Kernels with OI > ridge → COMPUTE BOUND")

---
## Section 9 — Achieved Performance (GFLOPs) vs Size

This shows how fast each kernel runs in terms of actual math throughput, compared to the theoretical peaks.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('⚡ Achieved GFLOPs vs Problem Size', fontsize=14, fontweight='bold')

for ax, kname in zip(axes, kernel_names):
    kdf = df[df['kernel'] == kname]
    
    for prec, color in kernel_colors.items():
        sub = kdf[kdf['precision'] == prec].sort_values('hidden_dim')
        ax.plot(sub['hidden_dim'], sub['achieved_gflops'],
                color=color, marker=markers[prec], linewidth=2, markersize=8, label=f'{prec} achieved')
    
    # Peak lines
    ax.axhline(y=PEAK_FP32_TFLOPS * 1000, color='#E74C3C', linestyle=':', alpha=0.8, linewidth=2,
               label=f'FP32 Peak ({PEAK_FP32_TFLOPS:.0f} TFLOP/s)')
    ax.axhline(y=PEAK_FP16_TFLOPS * 1000, color='#3498DB', linestyle=':', alpha=0.8, linewidth=2,
               label=f'FP16 Peak ({PEAK_FP16_TFLOPS:.0f} TFLOP/s)')
    
    ax.set_xlabel('Hidden Dimension D', fontsize=10)
    ax.set_ylabel('Achieved Performance (GFLOP/s)', fontsize=10)
    ax.set_title(f'{kname}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(SIZES)
    ax.set_xticklabels(SIZES, rotation=45)

plt.tight_layout()
plt.savefig('gflops_vs_size.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 10 — 🗺️ THE MAIN RESULT: Kernel Mapping on Roofline

This is the centerpiece figure of the entire project. We overlay all three kernels (at all sizes) onto the roofline chart.

**How to read this plot:**
- Each dot = one kernel at one size
- Dots near/on the **slanted line** = **memory-bound** (limited by how fast memory can be read)
- Dots near/on the **flat ceiling** = **compute-bound** (limited by the GPU's math units)
- The further a dot is **below** the roofline = the more room there is for optimization

In [ ]:
KERNEL_STYLES = {
    'GEMM':             {'color': '#E74C3C', 'marker': 'o', 'label': 'GEMM (Matrix Multiply)'},
    'Element-wise Add': {'color': '#3498DB', 'marker': 's', 'label': 'Element-wise Add'},
    'Softmax':          {'color': '#2ECC71', 'marker': '^', 'label': 'Softmax'},
}

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(f'🗺️ Kernel Roofline Mapping\n{gpu_name}', fontsize=14, fontweight='bold')

for ax_idx, (ax, prec_name, compute, color_mem, color_comp) in enumerate([
    (axes[0], 'FP32', PEAK_FP32_TFLOPS, '#E74C3C', '#FF7675'),
    (axes[1], 'FP16', PEAK_FP16_TFLOPS, '#0984E3', '#74B9FF'),
]):
    ax.set_xscale('log')
    ax.set_yscale('log')
    
    # ── Draw roofline ──
    oi_range   = np.logspace(-3, 4, 1000)
    comp_gflops = compute * 1000
    ridge_oi   = comp_gflops / PEAK_BW_GBs
    
    # Memory-bound segment
    oi_mem  = oi_range[oi_range <= ridge_oi]
    ax.plot(oi_mem, oi_mem * PEAK_BW_GBs, color=color_mem, linewidth=3,
            linestyle='-', label=f'Mem BW Roof ({PEAK_BW_GBs:.0f} GB/s)', zorder=1)
    
    # Compute ceiling
    oi_comp = oi_range[oi_range >= ridge_oi]
    ax.plot(oi_comp, np.full_like(oi_comp, comp_gflops), color=color_comp, linewidth=3,
            linestyle='-', label=f'{prec_name} Compute Roof ({compute:.0f} TFLOP/s)', zorder=1)
    
    # Ridge point
    ax.axvline(x=ridge_oi, color='gray', linestyle='--', alpha=0.5, linewidth=1.5)
    ax.scatter([ridge_oi], [comp_gflops], color='black', s=100, zorder=5)
    ax.annotate(f'Ridge: {ridge_oi:.1f}', xy=(ridge_oi, comp_gflops),
                xytext=(ridge_oi * 2, comp_gflops * 0.6),
                fontsize=8, arrowprops=dict(arrowstyle='->', lw=1),
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
    
    # ── Plot kernel points ──
    prec_df = df[df['precision'] == prec_name]
    
    for kname, kstyle in KERNEL_STYLES.items():
        kdf = prec_df[prec_df['kernel'] == kname].sort_values('hidden_dim')
        
        ax.scatter(kdf['operational_intensity'], kdf['achieved_gflops'],
                   color=kstyle['color'], marker=kstyle['marker'], s=120,
                   zorder=10, label=kstyle['label'], edgecolors='black', linewidths=0.8)
        
        # Draw trajectory line connecting sizes
        ax.plot(kdf['operational_intensity'], kdf['achieved_gflops'],
                color=kstyle['color'], alpha=0.4, linewidth=1.5, linestyle='--', zorder=5)
        
        # Label each size point
        for _, row in kdf.iterrows():
            ax.annotate(f"D={int(row['hidden_dim'])}",
                        xy=(row['operational_intensity'], row['achieved_gflops']),
                        xytext=(5, 5), textcoords='offset points',
                        fontsize=6.5, color=kstyle['color'], fontweight='bold')
    
    # Region labels
    ax.text(0.03, 0.15, '⬅ MEMORY\nBOUND', transform=ax.transAxes,
            fontsize=11, color=color_mem, fontweight='bold', alpha=0.7)
    ax.text(0.72, 0.15, 'COMPUTE\nBOUND ➡', transform=ax.transAxes,
            fontsize=11, color=color_comp, fontweight='bold', alpha=0.7)
    
    ax.set_xlabel('Operational Intensity (FLOP/Byte)', fontsize=11)
    ax.set_ylabel('Achieved Performance (GFLOP/s)', fontsize=11)
    ax.set_title(f'{prec_name} Precision Roofline Map', fontsize=12, fontweight='bold')
    ax.legend(loc='upper left', fontsize=8, framealpha=0.9)
    ax.grid(True, which='both', alpha=0.2)

plt.tight_layout()
plt.savefig('roofline_kernel_map.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Main roofline map saved as 'roofline_kernel_map.png'")

---
## Section 11 — FP32 vs FP16 Comparative Analysis

Do kernels change their classification (memory-bound vs compute-bound) when we switch from FP32 to FP16?

This is a key question: FP16 doubles the compute ceiling, but the memory bandwidth stays the same.
So FP16 only helps kernels that are near the compute ceiling — not memory-bound ones.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('📊 FP32 vs FP16 — Performance & OI Comparison per Kernel', fontsize=14, fontweight='bold')

bar_x   = np.arange(len(SIZES))
bar_w   = 0.35

for col, kname in enumerate(kernel_names):
    kdf     = df[df['kernel'] == kname]
    fp32_df = kdf[kdf['precision'] == 'FP32'].sort_values('hidden_dim')
    fp16_df = kdf[kdf['precision'] == 'FP16'].sort_values('hidden_dim')
    
    # ── Top row: GFLOPs comparison ──
    ax = axes[0, col]
    bars1 = ax.bar(bar_x - bar_w/2, fp32_df['achieved_gflops'], bar_w,
                   color='#E74C3C', label='FP32', alpha=0.85)
    bars2 = ax.bar(bar_x + bar_w/2, fp16_df['achieved_gflops'], bar_w,
                   color='#3498DB', label='FP16', alpha=0.85)
    ax.axhline(y=PEAK_FP32_TFLOPS * 1000, color='#E74C3C', linestyle='--', alpha=0.6, linewidth=1.5)
    ax.axhline(y=PEAK_FP16_TFLOPS * 1000, color='#3498DB', linestyle='--', alpha=0.6, linewidth=1.5)
    ax.set_xticks(bar_x)
    ax.set_xticklabels(SIZES)
    ax.set_xlabel('D=N (Size)')
    ax.set_ylabel('Achieved GFLOP/s')
    ax.set_title(f'{kname}\nAchieved Performance', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, axis='y', alpha=0.3)
    
    # ── Bottom row: OI comparison ──
    ax = axes[1, col]
    bars1 = ax.bar(bar_x - bar_w/2, fp32_df['operational_intensity'], bar_w,
                   color='#E74C3C', label='FP32 OI', alpha=0.85)
    bars2 = ax.bar(bar_x + bar_w/2, fp16_df['operational_intensity'], bar_w,
                   color='#3498DB', label='FP16 OI', alpha=0.85)
    ax.axhline(y=ridge_fp32, color='#E74C3C', linestyle='--', alpha=0.6, linewidth=1.5,
               label=f'FP32 Ridge ({ridge_fp32:.1f})')
    ax.axhline(y=ridge_fp16, color='#3498DB', linestyle='--', alpha=0.6, linewidth=1.5,
               label=f'FP16 Ridge ({ridge_fp16:.1f})')
    ax.set_xticks(bar_x)
    ax.set_xticklabels(SIZES)
    ax.set_xlabel('D=N (Size)')
    ax.set_ylabel('Operational Intensity (FLOP/Byte)')
    ax.set_title(f'{kname}\nOperational Intensity', fontweight='bold')
    ax.legend(fontsize=7)
    ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('fp32_vs_fp16_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 12 — Bound Classification Table

For each kernel at each size and precision, we classify it as **memory-bound** or **compute-bound**.
We also compute the **efficiency** — how close is the achieved performance to the theoretical maximum?

In [ ]:
def classify(oi, prec):
    """Classify kernel as memory-bound or compute-bound using ridge point."""
    ridge = ridge_fp32 if prec == 'FP32' else ridge_fp16
    return '💾 Memory-Bound' if oi < ridge else '⚙️ Compute-Bound'

def efficiency(achieved_gflops, oi, prec):
    """How much of the theoretical max are we achieving? (%)"""
    compute = PEAK_FP32_TFLOPS if prec == 'FP32' else PEAK_FP16_TFLOPS
    theoretical_max = min(compute * 1000, oi * PEAK_BW_GBs)
    return 100.0 * achieved_gflops / theoretical_max


print("═" * 105)
print("                    🏷️  KERNEL BOUND CLASSIFICATION TABLE")
print("═" * 105)

class_rows = []
for _, row in df.iterrows():
    bound   = classify(row['operational_intensity'], row['precision'])
    eff     = efficiency(row['achieved_gflops'], row['operational_intensity'], row['precision'])
    ridge   = ridge_fp32 if row['precision'] == 'FP32' else ridge_fp16
    class_rows.append([
        row['kernel'], row['precision'], int(row['hidden_dim']),
        f"{row['operational_intensity']:.3f}", f"{ridge:.2f}",
        f"{row['achieved_gflops']:.1f}", bound, f"{eff:.1f}%"
    ])

print(tabulate(class_rows,
               headers=['Kernel', 'Prec', 'D=N', 'OI (FLOP/B)', 'Ridge', 
                        'GFLOPs', 'Classification', 'Efficiency'],
               tablefmt='grid'))

# Summary statistics
print("\n📌 Summary:")
for kname in kernel_names:
    for prec in ['FP32', 'FP16']:
        sub = df[(df['kernel']==kname) & (df['precision']==prec)]
        all_bound = [classify(r['operational_intensity'], prec) for _, r in sub.iterrows()]
        unique    = set(all_bound)
        bound_str = 'Always Memory-Bound' if len(unique)==1 and '💾' in list(unique)[0] else \
                    'Always Compute-Bound' if len(unique)==1 else 'Mixed (shifts with size)'
        print(f"  {kname:<22} {prec}: {bound_str}")

---
## Section 13 — Scaling Study: Does Classification Shift with Size?

This is one of the most interesting findings: GEMM starts memory-bound at small sizes, but may shift to compute-bound at large sizes. This is because larger matrices have better data reuse — the GPU can amortize memory loads over more computations.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('📈 Scaling Study: How Bound Classification Changes with Size', 
             fontsize=13, fontweight='bold')

for ax, prec, compute, ridge_pt, cmap in [
    (axes[0], 'FP32', PEAK_FP32_TFLOPS, ridge_fp32, ['#E74C3C', '#3498DB', '#2ECC71']),
    (axes[1], 'FP16', PEAK_FP16_TFLOPS, ridge_fp16, ['#C0392B', '#1A5276', '#1D8348']),
]:
    ax.set_xscale('log')
    ax.set_yscale('log')
    
    oi_range    = np.logspace(-3, 4, 1000)
    comp_gflops = compute * 1000
    performance = np.minimum(comp_gflops, oi_range * PEAK_BW_GBs)
    ax.plot(oi_range, performance, 'k-', linewidth=2.5, alpha=0.6, label='Roofline', zorder=1)
    
    prec_df = df[df['precision'] == prec]
    for kname, color in zip(kernel_names, cmap):
        kdf = prec_df[prec_df['kernel'] == kname].sort_values('hidden_dim')
        
        # Color points by bound type
        for _, row in kdf.iterrows():
            is_comp_bound = row['operational_intensity'] >= ridge_pt
            edge = 'gold' if is_comp_bound else 'black'
            size = 180 if is_comp_bound else 100
            ax.scatter(row['operational_intensity'], row['achieved_gflops'],
                       color=color, s=size, edgecolors=edge, linewidths=2,
                       marker=KERNEL_STYLES[kname]['marker'], zorder=10)
            ax.annotate(f"{kname[:4]}\nD={int(row['hidden_dim'])}",
                        xy=(row['operational_intensity'], row['achieved_gflops']),
                        xytext=(4, 4), textcoords='offset points', fontsize=6, color=color)
        
        ax.plot(kdf['operational_intensity'], kdf['achieved_gflops'],
                color=color, linestyle='--', alpha=0.5, linewidth=1.5,
                label=kname, zorder=5)
    
    ax.axvline(x=ridge_pt, color='gray', linestyle=':', linewidth=2, alpha=0.7)
    ax.text(ridge_pt * 1.1, comp_gflops * 0.3, f'Ridge\n{ridge_pt:.1f}',
            fontsize=8, color='gray')
    
    # Legend for bound type
    from matplotlib.lines import Line2D
    custom = [
        Line2D([0],[0], marker='o', color='w', markerfacecolor='gray', markeredgecolor='gold', 
               markersize=10, label='Compute-Bound (gold edge)'),
        Line2D([0],[0], marker='o', color='w', markerfacecolor='gray', markeredgecolor='black',
               markersize=8, label='Memory-Bound (black edge)'),
    ]
    ax.legend(handles=custom + ax.get_legend_handles_labels()[0][1:], fontsize=7, loc='upper left')
    ax.set_xlabel('Operational Intensity (FLOP/Byte)', fontsize=11)
    ax.set_ylabel('Achieved GFLOP/s', fontsize=11)
    ax.set_title(f'{prec} — Scaling Classification', fontsize=12, fontweight='bold')
    ax.grid(True, which='both', alpha=0.2)

plt.tight_layout()
plt.savefig('scaling_classification.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 14 — PyTorch Profiler Analysis

The PyTorch Profiler is an **application-level** tool that tells us:
- Which GPU operations are taking the most time?
- What's the CPU vs GPU time breakdown?
- Are there any unexpected operations happening?

This validates our manual benchmarking and gives us operator-level attribution.

In [ ]:
from torch.profiler import profile, record_function, ProfilerActivity

# We profile at a representative medium size
PROF_D = 1024
PROF_N = 1024
PROF_B = 1

profiler_results = {}

for prec_name, (dtype, _) in PRECISIONS.items():
    print(f"\n{'='*60}")
    print(f"PyTorch Profiler — {prec_name} | D=N={PROF_D}")
    print('='*60)
    
    A     = torch.randn(PROF_B * PROF_N, PROF_D, dtype=dtype, device='cuda')
    W     = torch.randn(PROF_D, PROF_D, dtype=dtype, device='cuda')
    Add_A = torch.randn(PROF_B, PROF_N, PROF_D, dtype=dtype, device='cuda')
    Add_B = torch.randn(PROF_B, PROF_N, PROF_D, dtype=dtype, device='cuda')
    Smax  = torch.randn(PROF_B, PROF_N, PROF_N, dtype=dtype, device='cuda')
    
    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        record_shapes=True,
        with_flops=True,
        profile_memory=True,
    ) as prof:
        with record_function("GEMM_matmul"):
            for _ in range(10):
                with torch.no_grad():
                    _ = torch.matmul(A, W)
        torch.cuda.synchronize()
        
        with record_function("ElementwiseAdd"):
            for _ in range(10):
                with torch.no_grad():
                    _ = torch.add(Add_A, Add_B)
        torch.cuda.synchronize()
        
        with record_function("Softmax"):
            for _ in range(10):
                with torch.no_grad():
                    _ = torch.softmax(Smax, dim=-1)
        torch.cuda.synchronize()
    
    # Print top operations sorted by CUDA time
    print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=15))
    profiler_results[prec_name] = prof.key_averages()

print("\n✅ Profiler analysis complete.")
print("📌 Look at 'cuda_time_total' column — this shows actual GPU time per operation.")
print("📌 Compare 'cpu_time_total' vs 'cuda_time_total' — if CPU >> GPU, there's a CPU bottleneck.")

---
## Section 15 — Profiler Visual: CPU vs GPU Time Breakdown

Visualize the profiler results to see how much of the time each kernel spends on CPU vs GPU.

In [ ]:
# ── Safe attribute getter: handles different PyTorch versions ──
# Older PyTorch: cuda_time_total / cpu_time_total
# Newer PyTorch: self_device_time_total / self_cpu_time_total
def get_cuda_time(evt):
    """Safely get CUDA time from a profiler event regardless of PyTorch version."""
    for attr in ('cuda_time_total', 'self_device_time_total', 'device_time_total'):
        val = getattr(evt, attr, None)
        if val is not None:
            return val
    return 0

def get_cpu_time(evt):
    """Safely get CPU time from a profiler event regardless of PyTorch version."""
    for attr in ('cpu_time_total', 'self_cpu_time_total'):
        val = getattr(evt, attr, None)
        if val is not None:
            return val
    return 0

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('🔍 PyTorch Profiler — CPU vs CUDA Time per Operation',
             fontsize=13, fontweight='bold')

TARGET_KEYWORDS = ['matmul', 'aten::add', 'softmax', 'mm', 'gemm', 'add_']

for ax, prec_name in zip(axes, ['FP32', 'FP16']):
    events = profiler_results[prec_name]

    # Build rows using safe getters
    rows = []
    for evt in events:
        name_lower = evt.key.lower()
        cuda_t = get_cuda_time(evt)
        cpu_t  = get_cpu_time(evt)
        if any(kw in name_lower for kw in TARGET_KEYWORDS) and cuda_t > 0:
            rows.append({
                'name':    evt.key[:35],
                'cpu_ms':  cpu_t  / 1000,
                'cuda_ms': cuda_t / 1000,
            })

    # Fallback: top 8 ops by CUDA time if keyword filter finds nothing
    if not rows:
        all_with_cuda = [(evt, get_cuda_time(evt)) for evt in events if get_cuda_time(evt) > 0]
        all_with_cuda.sort(key=lambda x: x[1], reverse=True)
        rows = [{'name': e.key[:35], 'cpu_ms': get_cpu_time(e)/1000, 'cuda_ms': t/1000}
                for e, t in all_with_cuda[:8]]

    if rows:
        # Sort by CUDA time descending for cleaner plot
        rows.sort(key=lambda r: r['cuda_ms'], reverse=True)
        names   = [r['name']    for r in rows]
        cpu_ms  = [r['cpu_ms']  for r in rows]
        cuda_ms = [r['cuda_ms'] for r in rows]
        x = np.arange(len(names))
        w = 0.35

        ax.barh(x - w/2, cpu_ms,  w, label='CPU Time (ms)',  color='#E67E22', alpha=0.85)
        ax.barh(x + w/2, cuda_ms, w, label='CUDA Time (ms)', color='#8E44AD', alpha=0.85)
        ax.set_yticks(x)
        ax.set_yticklabels(names, fontsize=8)
        ax.set_xlabel('Time (ms)', fontsize=10)
        ax.set_title(f'{prec_name} Precision — Top Operations', fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, axis='x', alpha=0.3)
        ax.invert_yaxis()  # Biggest bar at top
    else:
        ax.text(0.5, 0.5, 'No GPU events captured.\nRe-run Section 14 first.',
                transform=ax.transAxes, ha='center', va='center', fontsize=12)
        ax.set_title(f'{prec_name} Precision', fontweight='bold')

plt.tight_layout()
plt.savefig('profiler_cpu_vs_gpu.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Profiler visualization saved as profiler_cpu_vs_gpu.png')

---
## Section 16 — Optimization Strategy Summary

Based on the roofline classification, we derive what optimization strategy makes sense for each kernel.
This is the key analysis section for your report.

In [ ]:
print("═" * 80)
print("     💡 OPTIMIZATION STRATEGY — Derived from Roofline Analysis")
print("═" * 80)

strategies = {
    'GEMM': {
        'bound': 'Compute-Bound (at large sizes)',
        'why': 'High OI due to weight matrix reuse across sequence positions. '
               'Each element of the weight matrix is reused for every sequence token, '
               'amortizing the memory load over many computations.',
        'strategies': [
            '✅ Use FP16/BF16 to double the compute ceiling (Tensor Cores)',
            '✅ Use cuBLAS / cuDNN optimized GEMM kernels',
            '✅ Improve tiling strategies (blocked matrix multiplication)',
            '✅ Increase batch size / sequence length to amortize launch overhead',
            '⚠️  Quantization (INT8/INT4) helps memory traffic at small sizes',
        ],
        'fp16_benefit': 'HIGH — shifts compute ceiling, can double throughput',
    },
    'Element-wise Add': {
        'bound': 'ALWAYS Memory-Bound (all sizes)',
        'why': 'Extremely low OI: does exactly 1 FLOP per element but must load '
               '2 arrays and write 1. The arithmetic intensity (0.083-0.167 FLOP/B) '
               'is far below the ridge point at every size.',
        'strategies': [
            '✅ Kernel fusion (fuse add with preceding/following op to avoid extra memory round-trips)',
            '✅ Reduce memory traffic (in-place operations where possible)',
            '✅ Optimize memory layout (contiguous tensors, coalesced access)',
            '⚠️  FP16 does NOT help the compute side — memory is the bottleneck',
            '✅ Quantization reduces bytes/element, directly helps bandwidth budget',
        ],
        'fp16_benefit': 'PARTIAL — reduces bytes per element (2 vs 4 bytes), helping bandwidth',
    },
    'Softmax': {
        'bound': 'Memory-Bound (all sizes)',
        'why': 'Softmax performs ~5 FLOPs per element (max, subtract, exp, sum, divide) '
               'but must pass over the data multiple times (two-pass algorithm). '
               'Its OI is low and dominated by the memory traffic of the attention matrix.',
        'strategies': [
            '✅ FlashAttention: fused kernel that keeps data in SRAM, avoids HBM round-trips',
            '✅ Online softmax (single-pass) reduces memory traffic',
            '✅ Kernel fusion with attention score computation',
            '⚠️  More FP16 compute ceiling does NOT help — memory is bottleneck',
            '✅ Tiling over the sequence dimension to fit in shared memory',
        ],
        'fp16_benefit': 'LOW for compute — but FP16 reduces memory footprint of attention matrix',
    },
}

for kname, info in strategies.items():
    print(f"\n{'─'*70}")
    print(f"  Kernel: {kname}")
    print(f"  Classification: {info['bound']}")
    print(f"  Why: {info['why']}")
    print(f"  FP16 Benefit: {info['fp16_benefit']}")
    print(f"  Optimization Strategies:")
    for s in info['strategies']:
        print(f"    {s}")

print(f"\n{'═'*80}")
print("KEY INSIGHT (from team shared specs):")
print("  'Roofline analysis reveals that not all LLM kernels benefit equally")
print("   from tensor core acceleration. While GEMM may shift under FP16,")
print("   element-wise operations remain memory-bound, implying HETEROGENEOUS")
print("   optimization strategies within a single Transformer block.'")
print('='*80)

---
## Section 17 — Final Summary Dashboard

One comprehensive visual summary of everything we found.

In [ ]:
fig = plt.figure(figsize=(20, 14))
fig.suptitle(f'📊 ECE-226 Benchmarking Dashboard\n{gpu_name}',
             fontsize=15, fontweight='bold', y=0.98)

gs = fig.add_gridspec(3, 3, hspace=0.45, wspace=0.35)

# ── Top row: Roofline maps ──
for col, (prec, compute) in enumerate([('FP32', PEAK_FP32_TFLOPS), ('FP16', PEAK_FP16_TFLOPS)]):
    ax = fig.add_subplot(gs[0, col])
    ax.set_xscale('log'); ax.set_yscale('log')
    oi_range    = np.logspace(-3, 4, 1000)
    comp_gflops = compute * 1000
    ridge_val   = comp_gflops / PEAK_BW_GBs
    ax.plot(oi_range, np.minimum(comp_gflops, oi_range * PEAK_BW_GBs),
            'k-', linewidth=2.5, alpha=0.7)
    prec_df = df[df['precision'] == prec]
    for kname, kstyle in KERNEL_STYLES.items():
        kdf = prec_df[prec_df['kernel'] == kname].sort_values('hidden_dim')
        ax.scatter(kdf['operational_intensity'], kdf['achieved_gflops'],
                   color=kstyle['color'], marker=kstyle['marker'], s=80,
                   zorder=10, label=kname[:4], edgecolors='black', linewidths=0.5)
        ax.plot(kdf['operational_intensity'], kdf['achieved_gflops'],
                color=kstyle['color'], alpha=0.3, linewidth=1)
    ax.set_xlabel('OI (FLOP/Byte)', fontsize=8)
    ax.set_ylabel('GFLOP/s', fontsize=8)
    ax.set_title(f'{prec} Roofline', fontsize=10, fontweight='bold')
    ax.legend(fontsize=7)
    ax.grid(True, which='both', alpha=0.2)

# ── Hardware specs box (top right) ──
ax_info = fig.add_subplot(gs[0, 2])
ax_info.axis('off')
info_text = (
    f"GPU: {gpu_name}\n\n"
    f"Peak FP32: {PEAK_FP32_TFLOPS:.1f} TFLOP/s\n"
    f"Peak FP16: {PEAK_FP16_TFLOPS:.1f} TFLOP/s\n"
    f"Bandwidth: {PEAK_BW_GBs:.0f} GB/s\n"
    f"VRAM:      {TOTAL_MEM_GB:.1f} GB\n\n"
    f"FP32 Ridge: {ridge_fp32:.2f} FLOP/B\n"
    f"FP16 Ridge: {ridge_fp16:.2f} FLOP/B\n\n"
    f"Sizes: {SIZES}\n"
    f"Batch: B={BATCH_SIZE}\n"
    f"Repeats: 3 × 100 iters"
)
ax_info.text(0.05, 0.95, info_text, transform=ax_info.transAxes,
             fontsize=9, verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
ax_info.set_title('Hardware & Config', fontsize=10, fontweight='bold')

# ── Middle row: Latency curves ──
for col, kname in enumerate(kernel_names):
    ax = fig.add_subplot(gs[1, col])
    kdf = df[df['kernel'] == kname]
    for prec, color in [('FP32', '#E74C3C'), ('FP16', '#3498DB')]:
        sub = kdf[kdf['precision'] == prec].sort_values('hidden_dim')
        ax.plot(sub['hidden_dim'], sub['latency_ms'], color=color,
                marker='o', linewidth=2, markersize=6, label=prec)
    ax.set_xlabel('D=N', fontsize=8)
    ax.set_ylabel('Latency (ms)', fontsize=8)
    ax.set_title(f'{kname}\nLatency', fontsize=9, fontweight='bold')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(SIZES)
    ax.set_xticklabels(SIZES, rotation=45, fontsize=7)

# ── Bottom row: Classification summary ──
ax_cls = fig.add_subplot(gs[2, :])
ax_cls.axis('off')

cls_table = []
for kname in kernel_names:
    for prec in ['FP32', 'FP16']:
        sub = df[(df['kernel']==kname)&(df['precision']==prec)]
        oi_min  = sub['operational_intensity'].min()
        oi_max  = sub['operational_intensity'].max()
        gf_min  = sub['achieved_gflops'].min()
        gf_max  = sub['achieved_gflops'].max()
        ridge   = ridge_fp32 if prec == 'FP32' else ridge_fp16
        bound   = 'Memory-Bound' if oi_max < ridge else ('Compute-Bound' if oi_min >= ridge else 'Shifts with size')
        opt     = ('Tensor Cores / FP16 / Tiling' if bound=='Compute-Bound' 
                   else 'Fusion / Quantization / Layout' if bound=='Memory-Bound'
                   else 'Both strategies apply')
        cls_table.append([kname, prec, f"{oi_min:.3f}–{oi_max:.3f}",
                          f"{gf_min:.0f}–{gf_max:.0f}", bound, opt])

tbl = ax_cls.table(
    cellText=cls_table,
    colLabels=['Kernel', 'Prec', 'OI Range (FLOP/B)', 'GFLOPs Range', 'Classification', 'Optimization'],
    cellLoc='center', loc='center', bbox=[0, 0, 1, 1]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)

# Color-code bound type
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2C3E50')
        cell.set_text_props(color='white', fontweight='bold')
    elif row > 0 and col == 4:
        txt = cell.get_text().get_text()
        if 'Memory' in txt:
            cell.set_facecolor('#AED6F1')
        elif 'Compute' in txt:
            cell.set_facecolor('#ABEBC6')
        else:
            cell.set_facecolor('#FAD7A0')
    elif row % 2 == 0:
        cell.set_facecolor('#F8F9FA')

ax_cls.set_title('🏷️  Bound Classification & Optimization Strategy Summary',
                 fontsize=11, fontweight='bold', pad=10)

plt.savefig('dashboard_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Dashboard saved as 'dashboard_summary.png'")

---
## Section 18 — Export All Results & File Summary

In [ ]:
# Save results CSV with all columns
df['bound_fp32'] = df.apply(lambda r: classify(r['operational_intensity'], 'FP32') 
                             if r['precision']=='FP32' else '', axis=1)
df['bound_fp16'] = df.apply(lambda r: classify(r['operational_intensity'], 'FP16') 
                             if r['precision']=='FP16' else '', axis=1)
df['efficiency_pct'] = df.apply(lambda r: efficiency(r['achieved_gflops'], 
                                                       r['operational_intensity'], r['precision']), axis=1)
df.to_csv('benchmark_results_full.csv', index=False)

print("═" * 55)
print("📁 FILES PRODUCED BY THIS NOTEBOOK")
print("═" * 55)
files = [
    ['benchmark_results_full.csv',   'Raw results — all kernels, all sizes, all precisions'],
    ['roofline_theoretical.png',     'Roofline plots — FP32 & FP16 theoretical limits'],
    ['latency_vs_size.png',          'Latency curves — FP32 vs FP16 per kernel'],
    ['oi_vs_size.png',               'Operational Intensity curves per kernel'],
    ['gflops_vs_size.png',           'Achieved GFLOPs vs size per kernel'],
    ['roofline_kernel_map.png',      '★ MAIN FIGURE: All kernels mapped onto roofline'],
    ['fp32_vs_fp16_comparison.png',  'Bar chart comparison FP32 vs FP16'],
    ['scaling_classification.png',   'Scaling study: how bound type shifts with size'],
    ['profiler_cpu_vs_gpu.png',      'PyTorch Profiler: CPU vs GPU time breakdown'],
    ['dashboard_summary.png',        'Full summary dashboard'],
]
print(tabulate(files, headers=['Filename', 'Description'], tablefmt='grid'))

print("\n📌 For your report, the most important figures are:")
print("   1. roofline_theoretical.png — Shows the hardware limits")
print("   2. roofline_kernel_map.png  — Shows where kernels sit on the roofline (Main figure)")
print("   3. scaling_classification.png — Shows how GEMM shifts with size")
print("   4. fp32_vs_fp16_comparison.png — Shows FP16 benefits per kernel")
print("   5. dashboard_summary.png — Good for poster presentation")

print(f"\n✅ Complete! Total experiments run: {len(df)}")
print(f"   GPU: {gpu_name}")
print(f"   Kernels: GEMM, Element-wise Add, Softmax")
print(f"   Sizes: {SIZES}")
print(f"   Precisions: FP32, FP16")